# Train the quantile transformer (Studio Lab, GPU runtime)
One walk-forward artifact per cell so a 4h session cutoff loses at most the epoch in flight — rerun the same cell with `--resume` after restarting the runtime.
Prereq (once per Studio Lab project): clone the repo and `pip install -e .` inside it.

In [ ]:
import os, pathlib
os.chdir(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "pyproject.toml").exists()))
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2022.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2023.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2024.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.eval.run --transformer-root models/transformer/v1 --out models/backtests/bakeoff.json
!git add models/transformer/v1 models/backtests configs
!git commit -m "model: transformer v1 walk-forward artifacts + bake-off results"
!git push